In [1]:
!pip install -U langchain langchain-google-genai langchain-chroma chromadb

In [2]:
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key

In [3]:
import os
from datetime import datetime

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage

#os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

In [4]:
# 1. LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

# 2. Embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# 3. ChromaDB vector database
documents = [
    Document(page_content="Machine learning allows computers to learn patterns from data."),
    Document(page_content="Supervised learning uses labelled data for training."),
    Document(page_content="Unsupervised learning finds hidden patterns in unlabelled data."),
    Document(page_content="Reinforcement learning learns by rewards and penalties."),
    Document(page_content="A neural network is inspired by the human brain and contains layers of neurons.")
]

vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="study_notes",
    persist_directory="./chroma_db"
)

retriever = vector_db.as_retriever(search_kwargs={"k": 2})

In [5]:
# 4. Tool
@tool
def get_current_time() -> str:
    """Returns the current date and time."""
    return str(datetime.now())


tools = {
    "get_current_time": get_current_time
}

In [6]:


# 5. Conversation memory
chat_history = []


def ask_agent(user_question):
    global chat_history

    # Retrieve related knowledge from ChromaDB
    docs = retriever.invoke(user_question)

    context = "\n".join([doc.page_content for doc in docs])

    memory_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            memory_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            memory_text += f"Assistant: {msg.content}\n"

    prompt = f"""
You are a helpful study assistant.

Use the following retrieved notes if relevant:

{context}

Conversation memory:

{memory_text}

User question:
{user_question}

Rules:
- Answer in simple English.
- If the question asks about current time, say: TOOL:get_current_time
- Otherwise answer normally.
"""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    # Simple tool calling
    if answer == "TOOL:get_current_time":
        tool_result = get_current_time.invoke({})
        final_prompt = f"""
The user asked: {user_question}

The tool returned:
{tool_result}

Now answer the user in simple English.
"""
        final_response = llm.invoke(final_prompt)
        answer = final_response.content

    # Save to memory
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))

    return answer

In [7]:
# 6. Chat loop
print("Study Agent Started. Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        break

    answer = ask_agent(question)
    print("\nAgent:", answer)
    print()

Study Agent Started. Type 'exit' to stop.

You: what is supervised learning?

Agent: Supervised learning is a type of machine learning where computers learn patterns from data that has been labelled. This means the training data includes the correct answers, which helps the computer learn what to look for.

You: What is the difference between supervised and unsupervised learning?

Agent: The main difference lies in the type of data they use:

*   **Supervised learning** uses **labelled data** for training. This means the data comes with the correct answers, helping the computer learn what to look for.
*   **Unsupervised learning** uses **unlabelled data**. It finds hidden patterns or structures in the data without any pre-existing correct answers.

You: what did i ask before

Agent: You previously asked: "What is the difference between supervised and unsupervised learning?"

You: what are the previous three questions i asked

Agent: You previously asked:

1.  "what did i ask before"
2.